In [1]:
import pandas as pd
import numpy as np
import re
import os

raw_data_path = "../data/raw/houses_raw.csv" 
cols = [
    'id', 'title', 'price_raw', 'area_raw', 'address_raw', 'url', 
    'seller_name', 'phone_number', 'bedrooms', 'bathrooms', 'floors', 
    'house_direction', 'legal_status', 'interior', 'ownership_type', 
    'price_trend', 'description', 'surrounding_area'
]

df = pd.read_csv(raw_data_path, names=cols, header=None)

print(f"Kích thước ma trận thô ban đầu: {df.shape[0]} dòng, {df.shape[1]} cột.")
df.head(5)

Kích thước ma trận thô ban đầu: 4991 dòng, 18 cột.


,id,title,price_raw,area_raw,address_raw,url,seller_name,phone_number,bedrooms,bathrooms,floors,house_direction,legal_status,interior,ownership_type,price_trend,description,surrounding_area
0,id,title,price_raw,area_raw,address_raw,url,seller_name,phone_number,bedrooms,bathrooms,floors,house_direction,legal_status,interior,ownership_type,price_trend,description,surrounding_area
1,1005858,"Bán căn 3PN tháp T5 - Masteri Thảo Điền, hỗ tr...",12 tỷ,99 m²,P. An Khánh (Quận 2 cũ),https://batdongsan.com.vn/ban-can-ho-chung-cu-...,Nguyễn Duy Hoài,NaN,3 phòng,2 phòng,NaN,Nam,Sổ đỏ/ Sổ hồng,Đầy đủ,NaN,NaN,Bán căn hộ 3PN Masteri Thảo Điền tháp T5 | Vie...,NaN
2,895451,Chính chủ bán đất view sông hiếm hoi xây biệt ...,"11,6 tỷ","142,6 m²",P. Hiệp Bình (TP. Thủ Đức cũ),https://batdongsan.com.vn/ban-dat-duong-quoc-l...,NaN,NaN,NaN,NaN,NaN,Tây - Bắc,Sổ đỏ/ Sổ hồng.,NaN,NaN,NaN,Mình chính chủ cần bán đất view sông hiếm hoi ...,NaN
3,1288513,Tấn Trường: Bán 2PN hoa hậu trục B-01 + B.02 -...,39 tỷ,65 m²,P. An Khánh (Quận 2 cũ),https://batdongsan.com.vn/ban-can-ho-chung-cu-...,Nguyễn Tấn Trường,NaN,2 phòng,NaN,NaN,NaN,Có sổ,Nội thất cơ bản.,NaN,NaN,Tấn Trường: Bán 2PN Hoa hậu trục B - 01 + B. 0...,NaN
4,762832,"Siêu đẹp! Đất MT kinh doanh đường 20m, 7x18m n...",21 tỷ,123 m²,P. Hiệp Bình (TP. Thủ Đức cũ),https://batdongsan.com.vn/ban-dat-duong-12-phu...,NaN,NaN,NaN,NaN,NaN,Đông - Nam,Sổ đỏ,NaN,NaN,NaN,"Vị trí khu bên sông, Phường Hiệp Bình Chánh.Di...",NaN


In [2]:
# 1. In nhanh các dòng có giá tiền bị lỗi dính nhiều dấu phân tách (ví dụ: x.xxx.xx hoặc x.xxx,xx)
err_price = df[df['price_raw'].astype(str).str.contains(r'\d+[\.,]\d+[\.,]\d+', regex=True)]
print(f"{len(err_price)} DÒNG LỖI ĐỊNH DẠNG GIÁ TIỀN:\n", err_price['price_raw'].unique())

# 2. Ép kiểu an toàn để in các dòng có diện tích lớn bất thường (> 1000m²)
err_area = df[df['area_raw'].astype(str).str.contains(r'\d+[\.,]\d+[\.,]\d+', regex=True)]
print(f"{len(err_area)} DÒNG LỖI ĐỊNH DẠNG DIỆN TÍCH:\n", err_area['area_raw'].unique())

1 DÒNG LỖI ĐỊNH DẠNG GIÁ TIỀN:
 ['1.738,05 tỷ']
33 DÒNG LỖI ĐỊNH DẠNG DIỆN TÍCH:
 ['4.335,7 m²' '3.557,9 m²' '1.000,3 m²' '6.547,6 m²' '1.786,2 m²'
 '1.043,2 m²' '1.450,7 m²' '3.569,5 m²' '2.553,6 m²' '1.003,7 m²'
 '1.372,7 m²' '1.158,7 m²' '2.487,7 m²' '1.117,8 m²' '1.709,8 m²'
 '1.190,4 m²' '2.056,7 m²' '7.168,4 m²' '4.199,7 m²' '2.772,2 m²'
 '3.869,5 m²' '1.190,5 m²' '1.260,8 m²' '1.087,2 m²' '1.124,7 m²'
 '40.519,8 m²' '2.782,5 m²' '1.394,1 m²' '1.098,4 m²' '1.544,1 m²']


In [3]:
df.drop(columns=['id'], errors='ignore', inplace=True)
df = df.iloc[1:].reset_index(drop=True)
df = df[df['price_raw'].astype(str).str.strip() != '1.738,05 tỷ'].reset_index(drop=True)
df = df[~df['area_raw'].astype(str).str.contains(r'\d+\.\d+,', regex=True)].reset_index(drop=True)
print("Đã xóa hoàn toàn biến 'id' và dòng label.")

Đã xóa hoàn toàn biến 'id' và dòng label.


In [4]:
def classify_url_segment(url_str):
    if pd.isna(url_str):
        return 1 
    
    url_lower = str(url_str).lower()
    if 'ban-dat' in url_lower:
        return 0 # Đất
    elif 'chung-cu' in url_lower or 'can-ho' in url_lower:
        return 2 # Chung cư / Căn hộ
    
    return 1 # Nhà riêng / Nhà phố 

df['url'] = df['url'].apply(classify_url_segment)

print("Đã phân loại 'url' thành 3 phân khúc chiến lược (0: Đất, 1: Nhà, 2: Chung cư).")
print(df['url'].value_counts())

Đã phân loại 'url' thành 3 phân khúc chiến lược (0: Đất, 1: Nhà, 2: Chung cư).
url
1    2542
0    1887
2     528
Name: count, dtype: int64


In [5]:
def extract_property_type(row):
    url_segment = row['url']
    title_str = row['title']
    
    # TH1: URL là Đất, gán ngay là Đất
    if url_segment == 0:
        return 'Đất nền / Đất thổ cư'
        
    # TH2: URL là Chung cư, gán ngay là Chung cư / Căn hộ
    if url_segment == 2:
        return 'Chung cư / Căn hộ'
        
    # TH3: URL là Nhà, gán ngay là Nhà riêng
    if pd.isna(title_str):
        return 'Nhà riêng'
    
    return 'Nhà riêng'

df['title'] = df.apply(extract_property_type, axis=1)

print("Đã bóc tách 'title' chính xác dựa trên cấu trúc 3 phân khúc URL.")
print(df['title'].value_counts())

Đã bóc tách 'title' chính xác dựa trên cấu trúc 3 phân khúc URL.
title
Nhà riêng               2542
Đất nền / Đất thổ cư    1887
Chung cư / Căn hộ        528
Name: count, dtype: int64


In [6]:
def clean_price(price_str):
    if pd.isna(price_str) or 'thỏa thuận' in str(price_str).lower():
        return np.nan
    
    # Viết thường, xóa khoảng trắng và đổi dấu phẩy thành dấu chấm toán học
    price_str = str(price_str).strip().lower().replace(',', '.')
    
    # Tiến hành trích xuất số học bằng Regex
    match_ty = re.search(r'([\d.]+)\s*tỷ', price_str)
    if match_ty:
        return float(match_ty.group(1))
        
    match_trieu = re.search(r'([\d.]+)\s*triệu', price_str)
    if match_trieu:
        return float(match_trieu.group(1)) / 1000.0
    
    match_num = re.search(r'([\d.]+)', price_str)
    return float(match_num.group(1)) if match_num else np.nan

df['price_raw'] = df['price_raw'].apply(clean_price)

print("Đã lọc số gọn cho cột 'price_raw' (Đơn vị tính: Tỷ VNĐ).")
print(df['price_raw'].describe())

Đã lọc số gọn cho cột 'price_raw' (Đơn vị tính: Tỷ VNĐ).
count    4957.000000
mean       12.820675
std        23.447771
min         0.299000
25%         4.500000
50%         7.250000
75%        13.000000
max       450.000000
Name: price_raw, dtype: float64


In [7]:
def clean_area(area_str):
    if pd.isna(area_str):
        return np.nan
    
    # Viết thường, xóa khoảng trắng và đổi dấu phẩy thành dấu chấm toán học
    area_str = str(area_str).strip().lower().replace(',', '.')
    
    # Tiến hành trích xuất số học bằng Regex
    match_area = re.search(r'([\d.]+)', area_str)
    return float(match_area.group(1)) if match_area else np.nan

df['area_raw'] = df['area_raw'].apply(clean_area)

print("[CẬP NHẬT] Đã lọc số gọn cho cột 'area_raw' (Đơn vị tính: m2).")
print(df['area_raw'].describe())

[CẬP NHẬT] Đã lọc số gọn cho cột 'area_raw' (Đơn vị tính: m2).
count    4957.000000
mean      117.697671
std       121.977930
min         1.000000
25%        56.400000
50%        80.000000
75%       121.000000
max       996.700000
Name: area_raw, dtype: float64


In [8]:
print("Giữ nguyên chuỗi vị trí thô 'address_raw' phục vụ bóc tách địa chính.")
print(df['address_raw'].head(3))

Giữ nguyên chuỗi vị trí thô 'address_raw' phục vụ bóc tách địa chính.
0          P. An Khánh (Quận 2 cũ)
1    P. Hiệp Bình (TP. Thủ Đức cũ)
2          P. An Khánh (Quận 2 cũ)
Name: address_raw, dtype: object


In [9]:
def clean_to_integer(val):
    if pd.isna(val):
        return np.nan
    match = re.search(r'(\d+)', str(val))
    return int(match.group(1)) if match else np.nan

df['bedrooms'] = df['bedrooms'].apply(clean_to_integer)

print("Đã chuyển đổi cột 'bedrooms' về dạng số nguyên.")
print(df['bedrooms'].value_counts(dropna=False).head(5))

Đã chuyển đổi cột 'bedrooms' về dạng số nguyên.
bedrooms
NaN    2500
2.0     607
4.0     585
3.0     533
5.0     257
Name: count, dtype: int64


In [10]:
df['bathrooms'] = df['bathrooms'].apply(clean_to_integer)

print("Đã chuyển đổi cột 'bathrooms' về dạng số nguyên.")
print(df['bathrooms'].value_counts(dropna=False).head(5))

Đã chuyển đổi cột 'bathrooms' về dạng số nguyên.
bathrooms
NaN    2622
2.0     697
3.0     378
5.0     371
4.0     310
Name: count, dtype: int64


In [11]:
df['floors'] = df['floors'].apply(clean_to_integer)

print("Đã chuyển đổi cột 'floors' về dạng số nguyên.")
print(df['floors'].value_counts(dropna=False).head(5))

Đã chuyển đổi cột 'floors' về dạng số nguyên.
floors
NaN    2757
4.0     628
2.0     574
3.0     475
5.0     278
Name: count, dtype: int64


In [12]:
print("Giữ nguyên trường phân loại hướng 'house_direction'.")
print(df['house_direction'].value_counts(dropna=False).head(5))

Giữ nguyên trường phân loại hướng 'house_direction'.
house_direction
NaN           3683
Đông - Nam     275
Đông - Bắc     172
Tây - Bắc      171
Tây - Nam      156
Name: count, dtype: int64


In [13]:
if 'legal_status' in df.columns:
    df['legal_status'] = (
        df['legal_status']
        .astype(str)                 
        .str.strip()                 
        .str.rstrip('.')             
        .replace('nan', np.nan)      
    )

print(" Đã chuẩn hóa và làm sạch trường pháp lý 'legal_status'.")
print(df['legal_status'].value_counts(dropna=False).head(5))

 Đã chuẩn hóa và làm sạch trường pháp lý 'legal_status'.
legal_status
Sổ đỏ/ Sổ hồng      3676
NaN                  509
Sổ hồng              202
Có sổ                113
Hợp đồng mua bán      86
Name: count, dtype: int64


In [14]:
print("Giữ nguyên phân loại nội thất 'interior'.")
print(df['interior'].value_counts(dropna=False).head(5))

Giữ nguyên phân loại nội thất 'interior'.
interior
NaN               3185
Đầy đủ             888
Cơ bản             451
Không nội thất      86
Đầy đủ.             77
Name: count, dtype: int64


In [15]:
def clean_trend(trend_str):
    if pd.isna(trend_str):
        return np.nan
    match = re.search(r'(-?[\d.]+)', str(trend_str))
    return float(match.group(1)) if match else np.nan

df['price_trend'] = df['price_trend'].apply(clean_trend)

print("[CẬP NHẬT] Đã chuyển đổi cột 'price_trend' thành số thực.")
print(df['price_trend'].describe())

[CẬP NHẬT] Đã chuyển đổi cột 'price_trend' thành số thực.
count    0.0
mean     NaN
std      NaN
min      NaN
25%      NaN
50%      NaN
75%      NaN
max      NaN
Name: price_trend, dtype: float64


In [16]:
redundant_cols = ['seller_name', 'phone_number', 'price_trend', 'ownership_type', 'description', 'surrounding_area']
df.drop(columns=redundant_cols, errors='ignore', inplace=True)

print(f"Đã tiến hành xóa hàng loạt các cột rác và không sử dụng: {redundant_cols}")

Đã tiến hành xóa hàng loạt các cột rác và không sử dụng: ['seller_name', 'phone_number', 'price_trend', 'ownership_type', 'description', 'surrounding_area']


In [17]:
print("BẢNG DỮ LIỆU PHẲNG SAU KHI XỬ LÝ")
print(f"Kích thước DataFrame hiện tại: {df.shape[0]} dòng, {df.shape[1]} cột.")
print("\n")
print("Danh sách các cột hiện tại trong bộ nhớ:")
print(list(df.columns))
print("\n")
print("Tỷ lệ khuyết thiếu (%) trên từng cột sạch:")
print(df.isnull().mean() * 100)
print("\n")

output_dir = "../data/raw"
os.makedirs(output_dir, exist_ok=True)

output_path = os.path.join(output_dir, "01_text_parsing_and_feature_enhancement.csv")
df.to_csv(output_path, index=False)

print(f"XỬ LÝ THÀNH CÔNG CHẶNG 1! File sạch bề nổi đã được lưu tại: {output_path}")
df.head(5)

BẢNG DỮ LIỆU PHẲNG SAU KHI XỬ LÝ
Kích thước DataFrame hiện tại: 4957 dòng, 11 cột.


Danh sách các cột hiện tại trong bộ nhớ:
['title', 'price_raw', 'area_raw', 'address_raw', 'url', 'bedrooms', 'bathrooms', 'floors', 'house_direction', 'legal_status', 'interior']


Tỷ lệ khuyết thiếu (%) trên từng cột sạch:
title               0.000000
price_raw           0.000000
area_raw            0.000000
address_raw         0.000000
url                 0.000000
bedrooms           50.433730
bathrooms          52.894896
floors             55.618318
house_direction    74.298971
legal_status       10.268307
interior           64.252572
dtype: float64




XỬ LÝ THÀNH CÔNG CHẶNG 1! File sạch bề nổi đã được lưu tại: ../data/raw\01_text_parsing_and_feature_enhancement.csv


,title,price_raw,area_raw,address_raw,url,bedrooms,bathrooms,floors,house_direction,legal_status,interior
0,Chung cư / Căn hộ,12.00,99.0,P. An Khánh (Quận 2 cũ),2,3.0,2.0,NaN,Nam,Sổ đỏ/ Sổ hồng,Đầy đủ
1,Đất nền / Đất thổ cư,11.60,142.6,P. Hiệp Bình (TP. Thủ Đức cũ),0,NaN,NaN,NaN,Tây - Bắc,Sổ đỏ/ Sổ hồng,NaN
2,Chung cư / Căn hộ,39.00,65.0,P. An Khánh (Quận 2 cũ),2,2.0,NaN,NaN,NaN,Có sổ,Nội thất cơ bản.
3,Đất nền / Đất thổ cư,21.00,123.0,P. Hiệp Bình (TP. Thủ Đức cũ),0,NaN,NaN,NaN,Đông - Nam,Sổ đỏ,NaN
4,Nhà riêng,5.95,48.0,P. Tân Sơn Nhì (Q. Tân Phú cũ),1,2.0,2.0,2.0,Đông - Nam,Sổ đỏ/ Sổ hồng,Cơ bản
